# EGRAT: Edge-aware Graph-temporal Risk-Adaptive Trust Model
## Full IIoT security implementation

This notebook implements the requested pipeline:

1. TON_IoT acquisition and inspection
2. Missing-value handling, duplicate removal, robust scaling, categorical encoding, timestamp alignment, sliding windows and chronological splitting
3. Device communication graph construction
4. Edge-Conditioned GraphSAGE + depthwise separable dilated TCN
5. Evidence-Weighted Adaptive Trust Assessment (EWATA)
6. Risk-Adaptive Zero-Trust Access Enforcement (RAZTAE)
7. Comparative evaluation, ablation, external X-IIoTID validation and efficiency analysis

**Dataset sources**
- TON_IoT (primary): https://research.unsw.edu.au/projects/toniot-datasets
- X-IIoTID (external validation): https://ieeedataport.org/documents/x-iiotid-connectivity-and-device-agnostic-intrusion-datasetindustrial-internet-things

The official TON_IoT collection includes network traffic, telemetry and operating-system data. This notebook starts with the processed network-flow CSV and can merge telemetry/log CSVs when supplied. Download archives from the official source and place the extracted CSV files in the paths below. X-IIoTID access may require IEEE DataPort account access.

**Important:** Metrics are calculated from the files actually loaded and trained in this notebook. No performance values are fabricated. The external dataset is evaluated only after its columns and labels are mapped; it is not used to fit the primary model.

In [ ]:
# Install dependencies (run once in a fresh Colab/local environment)
%pip install -q numpy pandas scikit-learn matplotlib seaborn torch joblib

In [ ]:
from pathlib import Path
import os, re, json, time, copy, warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_auc_score, matthews_corrcoef,
    balanced_accuracy_score)
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ROOT = Path('Vanitha mam IIoT 1')
TON_DIR = ROOT / 'data' / 'raw' / 'ton_iot'
XIIOT_DIR = ROOT / 'data' / 'raw' / 'x_iiotid'
RESULT_DIR = ROOT / 'results'
for p in [TON_DIR, XIIOT_DIR, RESULT_DIR]: p.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('TON_IoT CSV directory:', TON_DIR.resolve())
print('X-IIoTID CSV directory:', XIIOT_DIR.resolve())

## 1. Dataset placement and acquisition

Download the processed **network** CSV files from the official TON_IoT page and extract them into `Vanitha mam IIoT 1/data/raw/ton_iot/`. The loader searches recursively, so the archive's internal folder structure can be retained.

For external validation, download X-IIoTID from IEEE DataPort and extract its CSV files into `Vanitha mam IIoT 1/data/raw/x_iiotid/`.

Optional TON_IoT telemetry and Windows/Linux event CSV files can also be placed under `ton_iot/telemetry/` and `ton_iot/logs/`. The network-flow table remains the supervised source; telemetry/log values are joined by device and nearest timestamp where compatible fields exist.

In [ ]:
def read_csv_tree(folder):
    files = sorted(Path(folder).rglob('*.csv'))
    if not files:
        raise FileNotFoundError(f'No CSV files found under {folder}. Download and extract the dataset first.')
    frames = []
    for f in files:
        try:
            d = pd.read_csv(f, low_memory=False)
            d['_source_file'] = f.name
            frames.append(d)
            print(f'{f.name}: {d.shape}')
        except Exception as e:
            print('Skipped', f, e)
    if not frames: raise ValueError(f'No readable CSV files in {folder}')
    return pd.concat(frames, ignore_index=True, sort=False)

ton_raw = read_csv_tree(TON_DIR)
print('Combined shape:', ton_raw.shape)
display(ton_raw.head())
display(pd.DataFrame({'column': ton_raw.columns, 'dtype': ton_raw.dtypes.astype(str)}))

In [ ]:
def find_col(df, candidates):
    low = {str(c).strip().lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in low: return low[name.lower()]
    for c in df.columns:
        cl = str(c).lower()
        if any(k in cl for k in candidates): return c
    return None

def normalise_label(v):
    s = str(v).strip().lower()
    if s in {'0','normal','benign','false','no','legitimate'}: return 'normal'
    if s in {'1','attack','anomaly','malicious','true','yes'}: return 'attack'
    return s if s not in {'nan','none',''} else 'unknown'

TS_COL = find_col(ton_raw, ['ts','timestamp','time'])
SRC_COL = find_col(ton_raw, ['src_ip','source_ip','srcip','source address','src'])
DST_COL = find_col(ton_raw, ['dst_ip','destination_ip','dstip','destination address','dst'])
BINARY_COL = find_col(ton_raw, ['label','binary_label','is_attack'])
TYPE_COL = find_col(ton_raw, ['type','attack','attack_type','category','class'])

if BINARY_COL is None and TYPE_COL is None:
    raise ValueError('Could not identify a target label. Inspect the columns and set BINARY_COL or TYPE_COL manually.')
if BINARY_COL is not None:
    ton_raw['_binary_label'] = ton_raw[BINARY_COL].map(normalise_label)
elif TYPE_COL is not None:
    ton_raw['_binary_label'] = ton_raw[TYPE_COL].map(normalise_label).apply(lambda x: 'normal' if x == 'normal' else 'attack')
if TYPE_COL is not None:
    ton_raw['_attack_type'] = ton_raw[TYPE_COL].astype(str).str.lower().replace({'nan':'unknown'})
else:
    ton_raw['_attack_type'] = ton_raw['_binary_label']
ton_raw['_attack_type'] = np.where(ton_raw['_binary_label'].eq('normal'), 'normal', ton_raw['_attack_type'])
print('Detected columns:', {'timestamp':TS_COL, 'source':SRC_COL, 'destination':DST_COL, 'binary_label':BINARY_COL, 'attack_type':TYPE_COL})
display(ton_raw['_attack_type'].value_counts(dropna=False).to_frame('records'))

## 2. Preprocessing and feature engineering

The target is kept separate from input features to prevent label leakage. Identifiers are retained for graph construction but excluded from numeric model features. Rows are sorted chronologically before splitting. Robust scaling is fitted on training data only.

In [ ]:
# Canonicalise timestamps and endpoint identifiers
df = ton_raw.copy()
if TS_COL is not None:
    if pd.api.types.is_numeric_dtype(df[TS_COL]):
        # TON_IoT timestamps are commonly Unix seconds; handle millisecond values too.
        vals = pd.to_numeric(df[TS_COL], errors='coerce')
        unit = 'ms' if vals.dropna().median() > 1e11 else 's'
        df['_time'] = pd.to_datetime(vals, unit=unit, errors='coerce', utc=True)
    else:
        df['_time'] = pd.to_datetime(df[TS_COL], errors='coerce', utc=True)
else:
    df['_time'] = pd.NaT

df['_src'] = df[SRC_COL].astype(str) if SRC_COL else 'device_unknown'
df['_dst'] = df[DST_COL].astype(str) if DST_COL else 'peer_unknown'
df['_src'] = df['_src'].replace({'nan':'device_unknown','None':'device_unknown'})
df['_dst'] = df['_dst'].replace({'nan':'peer_unknown','None':'peer_unknown'})
df['_binary_label'] = df['_binary_label'].replace({'unknown':'attack'})
df = df.drop_duplicates().reset_index(drop=True)
df = df.drop(columns=['_source_file'], errors='ignore')

# Select numeric and categorical predictors, excluding labels, timestamps and raw endpoint IDs.
exclude = {BINARY_COL, TYPE_COL, TS_COL, SRC_COL, DST_COL, '_binary_label', '_attack_type', '_time', '_src', '_dst'}
exclude = {x for x in exclude if x is not None}
numeric_cols = [c for c in df.select_dtypes(include=np.number).columns if c not in exclude]
categorical_cols = [c for c in df.select_dtypes(include=['object','category','bool']).columns if c not in exclude]
# Exclude high-cardinality free text fields to keep the model lightweight.
categorical_cols = [c for c in categorical_cols if df[c].nunique(dropna=True) <= 100]
print('Numeric features:', len(numeric_cols), numeric_cols[:20])
print('Categorical features:', len(categorical_cols), categorical_cols[:20])

# Impute numeric values with training-independent medians for initial data integrity;
# scaler itself is fitted only on the chronological training portion below.
for c in numeric_cols: df[c] = pd.to_numeric(df[c], errors='coerce')
df[numeric_cols] = df[numeric_cols].replace([np.inf,-np.inf], np.nan)
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median(numeric_only=True)).fillna(0)
for c in categorical_cols: df[c] = df[c].astype('string').fillna('missing').astype(str)

if df['_time'].notna().any():
    df = df.sort_values('_time', kind='stable').reset_index(drop=True)
else:
    warnings.warn('No usable timestamp detected; row order is used as a fallback chronological order.')

# Encode categorical predictors; stable integer codes are used for this dataset.
for c in categorical_cols:
    df[c] = pd.factorize(df[c], sort=True)[0].astype(np.float32)

feature_cols = numeric_cols + categorical_cols
if not feature_cols: raise ValueError('No usable predictor columns detected.')
X_all = df[feature_cols].to_numpy(dtype=np.float32)
y_names = df['_attack_type'].astype(str).to_numpy()
classes = sorted(pd.unique(y_names))
class_to_id = {c:i for i,c in enumerate(classes)}
y_all = np.array([class_to_id[v] for v in y_names], dtype=np.int64)
normal_id = class_to_id.get('normal', 0)
print('Classes:', class_to_id, '| rows:', len(df), '| features:', X_all.shape[1])

In [ ]:
# Chronological split: 70% train, 15% validation, 15% test.
n = len(df)
i_train, i_val = int(n*.70), int(n*.85)
train_idx = np.arange(0, i_train)
val_idx = np.arange(i_train, i_val)
test_idx = np.arange(i_val, n)

scaler = RobustScaler()
X_all[train_idx] = scaler.fit_transform(X_all[train_idx])
X_all[val_idx] = scaler.transform(X_all[val_idx])
X_all[test_idx] = scaler.transform(X_all[test_idx])
X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

print('Chronological split:', len(train_idx), len(val_idx), len(test_idx))
print('Train label distribution:')
display(pd.Series(y_names[train_idx]).value_counts().to_frame('records'))

## 3. Temporal device communication graph construction

Each industrial endpoint is a graph node. Each observed source-to-destination flow is a directed edge. Edge attributes include the available numeric flow measurements. Node attributes are the mean incoming/outgoing flow feature vector for that device in the time window. Attack labels are aggregated at source-device level within each window.

For manageable memory use, `MAX_DEVICES` and `MAX_WINDOWS` can be reduced on large datasets. These limits are applied after chronological sorting.

In [ ]:
WINDOW_SECONDS = 60
MAX_DEVICES = 3000
MAX_WINDOWS = 2500
MAX_EDGES_PER_WINDOW = 20000
TOP_K_NEIGHBOURS = 20

# Device vocabulary is built from the full endpoint identity field only; it is not a predictive feature.
device_values = pd.Index(pd.concat([df['_src'], df['_dst']]).astype(str).unique())
if len(device_values) > MAX_DEVICES:
    degree = pd.concat([df['_src'], df['_dst']]).value_counts()
    device_values = pd.Index(degree.head(MAX_DEVICES).index.astype(str))
device_to_id = {v:i for i,v in enumerate(device_values)}
src_ids = df['_src'].map(device_to_id).fillna(-1).astype(int).to_numpy()
dst_ids = df['_dst'].map(device_to_id).fillna(-1).astype(int).to_numpy()
keep = (src_ids >= 0) & (dst_ids >= 0)
df = df.loc[keep].reset_index(drop=True)
X_all, y_all, y_names = X_all[keep], y_all[keep], y_names[keep]
src_ids, dst_ids = src_ids[keep], dst_ids[keep]
train_idx = np.arange(0, int(len(df)*.70))
val_idx = np.arange(int(len(df)*.70), int(len(df)*.85))
test_idx = np.arange(int(len(df)*.85), len(df))

if df['_time'].notna().any():
    origin = df['_time'].min()
    win_ids = ((df['_time'] - origin).dt.total_seconds().fillna(0) // WINDOW_SECONDS).astype(int).to_numpy()
else:
    # Fallback: sequential blocks preserve order when timestamps are unavailable.
    win_ids = (np.arange(len(df)) // max(1, WINDOW_SECONDS)).astype(int)

unique_windows = np.unique(win_ids)
if len(unique_windows) > MAX_WINDOWS:
    chosen = set(unique_windows[:MAX_WINDOWS])
    m = np.isin(win_ids, list(chosen))
    df, X_all, y_all, y_names, src_ids, dst_ids, win_ids = [a[m] for a in [df, X_all, y_all, y_names, src_ids, dst_ids, win_ids]]
    train_idx = np.arange(0, int(len(df)*.70))
    val_idx = np.arange(int(len(df)*.70), int(len(df)*.85))
    test_idx = np.arange(int(len(df)*.85), len(df))

def make_snapshots(df, X, y, src, dst, win_ids, n_devices, max_edges=MAX_EDGES_PER_WINDOW, top_k=TOP_K_NEIGHBOURS):
    snapshots = []
    for w in np.unique(win_ids):
        ix = np.flatnonzero(win_ids == w)
        if len(ix) == 0: continue
        # Keep most active source devices in the window.
        counts = pd.Series(src[ix]).value_counts()
        active = set(counts.head(top_k * 100).index.tolist())
        ix = np.array([j for j in ix if src[j] in active and dst[j] in active], dtype=int)
        if len(ix) == 0: continue
        if len(ix) > max_edges:
            ix = ix[:max_edges]
        node_x = np.zeros((n_devices, X.shape[1]), dtype=np.float32)
        node_count = np.zeros(n_devices, dtype=np.float32)
        np.add.at(node_x, src[ix], X[ix]); np.add.at(node_count, src[ix], 1)
        np.add.at(node_x, dst[ix], X[ix]); np.add.at(node_count, dst[ix], 1)
        node_x /= np.maximum(node_count[:,None], 1)
        # Edge attributes are per-flow features; cap the edge feature dimension naturally at input width.
        edge_index = np.vstack([src[ix], dst[ix]]).astype(np.int64)
        edge_attr = X[ix].astype(np.float32)
        node_y = np.full(n_devices, -100, dtype=np.int64)
        # Source device gets the most severe observed label in the window (normal is class 0 where possible).
        for dev in np.unique(src[ix]):
            labels = y[ix[src[ix] == dev]]
            nonnormal = labels[labels != normal_id]
            node_y[dev] = int(pd.Series(nonnormal).value_counts().index[0]) if len(nonnormal) else normal_id
        snapshots.append({'window':int(w), 'x':node_x, 'edge_index':edge_index, 'edge_attr':edge_attr, 'y':node_y, 'node_count':node_count})
    return snapshots

snapshots = make_snapshots(df, X_all, y_all, src_ids, dst_ids, win_ids, len(device_values))
print('Devices:', len(device_values), '| graph snapshots:', len(snapshots))
print('Example graph:', snapshots[0]['x'].shape, snapshots[0]['edge_index'].shape, snapshots[0]['edge_attr'].shape)

## 4. EGRAT: Edge-Conditioned GraphSAGE–Depthwise Separable TCN

The spatial branch forms edge-conditioned messages from source-node and edge features, aggregates neighbour messages with mean aggregation, and updates node embeddings. The temporal branch applies depthwise separable dilated convolutions over each device's historical window sequence. The fused representation predicts the normal/attack category at device level.

In [ ]:
class EdgeConditionedSAGE(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden):
        super().__init__()
        self.msg = nn.Sequential(nn.Linear(node_dim + edge_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.self_lin = nn.Linear(node_dim, hidden)
        self.update = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(.2))
    def forward(self, x, edge_index, edge_attr):
        n = x.size(0)
        agg = torch.zeros((n, self.self_lin.out_features), device=x.device)
        deg = torch.zeros((n,1), device=x.device)
        if edge_index.numel():
            s, d = edge_index
            msg = self.msg(torch.cat([x[s], edge_attr], dim=1))
            agg.index_add_(0, d, msg)
            deg.index_add_(0, d, torch.ones((len(d),1), device=x.device))
        agg = agg / deg.clamp_min(1)
        return self.update(torch.cat([self.self_lin(x), agg], dim=1))

class DS_TCN_Block(nn.Module):
    def __init__(self, channels, dilation):
        super().__init__()
        self.depth = nn.Conv1d(channels, channels, 3, padding=2*dilation, dilation=dilation, groups=channels)
        self.point = nn.Conv1d(channels, channels, 1)
        self.norm = nn.BatchNorm1d(channels)
        self.act = nn.ReLU()
    def forward(self, x):
        z = self.point(self.depth(x))
        z = z[..., :x.size(-1)]
        return self.act(self.norm(z) + x)

class EGRAT(nn.Module):
    def __init__(self, node_dim, edge_dim, n_classes, hidden=64):
        super().__init__()
        self.sage = EdgeConditionedSAGE(node_dim, edge_dim, hidden)
        self.temporal_in = nn.Conv1d(node_dim, hidden, 1)
        self.tcn = nn.Sequential(DS_TCN_Block(hidden,1), DS_TCN_Block(hidden,2), DS_TCN_Block(hidden,4))
        self.classifier = nn.Sequential(nn.Linear(hidden*2, hidden), nn.ReLU(), nn.Dropout(.25), nn.Linear(hidden,n_classes))
    def forward(self, x, edge_index, edge_attr, history):
        spatial = self.sage(x, edge_index, edge_attr)
        temporal = self.tcn(self.temporal_in(history))[:,:,-1]
        return self.classifier(torch.cat([spatial, temporal], dim=1))

model = EGRAT(X_all.shape[1], X_all.shape[1], len(classes)).to(DEVICE)
print(model)
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
def to_tensor_snapshot(s):
    return (torch.tensor(s['x'], dtype=torch.float32, device=DEVICE),
            torch.tensor(s['edge_index'], dtype=torch.long, device=DEVICE),
            torch.tensor(s['edge_attr'], dtype=torch.float32, device=DEVICE),
            torch.tensor(s['y'], dtype=torch.long, device=DEVICE))

# Historical node features are aligned by the fixed device vocabulary.
def history_tensor(snap_list, t, seq_len=8):
    start = max(0, t-seq_len+1)
    arr = [s['x'] for s in snap_list[start:t+1]]
    if len(arr) < seq_len:
        arr = [np.zeros_like(arr[0]) for _ in range(seq_len-len(arr))] + arr
    return torch.tensor(np.stack(arr, axis=0).transpose(1,2,0), dtype=torch.float32, device=DEVICE)

# Map snapshots to chronological train/validation/test portions by snapshot index.
n_snap = len(snapshots)
tr_end, va_end = int(n_snap*.70), int(n_snap*.85)
train_snap_ids = list(range(0,tr_end))
val_snap_ids = list(range(tr_end,va_end))
test_snap_ids = list(range(va_end,n_snap))
print('Snapshot split:', len(train_snap_ids), len(val_snap_ids), len(test_snap_ids))

In [ ]:
# Class weights are computed only from training snapshots.
train_labels = np.concatenate([snapshots[i]['y'][snapshots[i]['y'] >= 0] for i in train_snap_ids]) if train_snap_ids else np.array([],dtype=int)
present = np.unique(train_labels)
weights = np.ones(len(classes), dtype=np.float32)
if len(present) > 1:
    cw = compute_class_weight('balanced', classes=present, y=train_labels)
    weights[present] = cw
criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=DEVICE), ignore_index=-100)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
EPOCHS = 20
PATIENCE = 5
best_val, best_state, wait = float('inf'), None, 0
history_log = {'train_loss':[], 'val_loss':[]}

def run_epoch(indices, training=False):
    model.train(training)
    losses = []
    context = torch.enable_grad() if training else torch.no_grad()
    with context:
        for t in indices:
            x, ei, ea, y = to_tensor_snapshot(snapshots[t])
            h = history_tensor(snapshots, t)
            logits = model(x, ei, ea, h)
            valid = y != -100
            if not valid.any(): continue
            loss = criterion(logits, y)
            if training:
                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses)) if losses else np.nan

for epoch in range(EPOCHS):
    tr_loss = run_epoch(train_snap_ids, True)
    va_loss = run_epoch(val_snap_ids, False)
    history_log['train_loss'].append(tr_loss); history_log['val_loss'].append(va_loss)
    print(f'Epoch {epoch+1:02d}/{EPOCHS} | train={tr_loss:.4f} | val={va_loss:.4f}')
    score = va_loss if np.isfinite(va_loss) else tr_loss
    if score < best_val:
        best_val, best_state, wait = score, copy.deepcopy(model.state_dict()), 0
    else:
        wait += 1
        if wait >= PATIENCE:
            print('Early stopping'); break
if best_state is not None: model.load_state_dict(best_state)

plt.figure(figsize=(8,4))
plt.plot(history_log['train_loss'], label='Training loss')
plt.plot(history_log['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.grid(alpha=.25); plt.tight_layout()
plt.savefig(RESULT_DIR/'egrat_training_curve.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Model evaluation

The evaluation is device-window classification. Only nodes with a label in the corresponding graph snapshot are included. The reported metrics are computed from the held-out chronological test snapshots.

In [ ]:
@torch.no_grad()
def collect_predictions(indices, trained_model=model):
    trained_model.eval()
    ys, preds, probs = [], [], []
    for t in indices:
        x, ei, ea, y = to_tensor_snapshot(snapshots[t])
        logits = trained_model(x, ei, ea, history_tensor(snapshots,t))
        valid = y != -100
        if valid.any():
            p = torch.softmax(logits[valid], dim=1).cpu().numpy()
            ys.extend(y[valid].cpu().numpy().tolist())
            preds.extend(p.argmax(1).tolist())
            probs.extend(p.tolist())
    return np.asarray(ys), np.asarray(preds), np.asarray(probs)

y_true, y_pred, y_prob = collect_predictions(test_snap_ids)
if len(y_true):
    p,r,f1,_ = precision_recall_fscore_support(y_true,y_pred,average='weighted',zero_division=0)
    metrics = {'Accuracy':accuracy_score(y_true,y_pred),'Precision_weighted':p,'Recall_weighted':r,
               'F1_weighted':f1,'Balanced_accuracy':balanced_accuracy_score(y_true,y_pred),
               'MCC':matthews_corrcoef(y_true,y_pred)}
    try: metrics['ROC_AUC_ovr'] = roc_auc_score(y_true,y_prob,multi_class='ovr',average='weighted')
    except Exception: metrics['ROC_AUC_ovr'] = np.nan
    display(pd.DataFrame(metrics,index=['EGRAT']).T.rename(columns={'EGRAT':'Value'}))
    print(classification_report(y_true,y_pred,labels=np.arange(len(classes)),target_names=classes,zero_division=0))
    cm = confusion_matrix(y_true,y_pred,labels=np.arange(len(classes)))
    plt.figure(figsize=(9,7)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=classes,yticklabels=classes)
    plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('EGRAT confusion matrix'); plt.tight_layout()
    plt.savefig(RESULT_DIR/'egrat_confusion_matrix.png',dpi=300,bbox_inches='tight'); plt.show()
else:
    print('No labelled test nodes were available. Inspect label mapping and graph construction.')

## 6. EWATA: Evidence-Weighted Adaptive Trust Assessment

EWATA combines model attack confidence, behavioural deviation, and communication reliability. Recent evidence receives greater weight through exponential time decay. Repeated normal observations gradually recover trust. The thresholds below are configurable policy parameters, not learned labels.

In [ ]:
def ewata_update(previous_trust, attack_probability, anomaly_score, communication_reliability,
                 dt=1.0, decay=0.15, recovery=0.04):
    # All evidence inputs are clipped to [0,1].
    ap = float(np.clip(attack_probability,0,1))
    an = float(np.clip(anomaly_score,0,1))
    cr = float(np.clip(communication_reliability,0,1))
    evidence_risk = 0.50*ap + 0.30*an + 0.20*(1-cr)
    decayed = previous_trust * np.exp(-decay*max(dt,0)*evidence_risk)
    if ap < 0.20 and an < 0.25 and cr > 0.75:
        decayed = decayed + recovery*(1-decayed)
    trust = float(np.clip(decayed,0,1))
    state = 'trusted' if trust >= 0.70 else ('suspicious' if trust >= 0.35 else 'untrusted')
    return trust, state, evidence_risk

# Generate trust histories from held-out predictions where available.
trust_records = []
if len(y_true):
    for i,(yt,yp,pr) in enumerate(zip(y_true,y_pred,y_prob)):
        attack_prob = 1.0-float(pr[normal_id]) if normal_id < len(pr) else float(yp != normal_id)
        # Confidence-derived anomaly score; edge/traffic statistics are not re-labelled.
        anomaly = attack_prob
        trust, state, risk = ewata_update(1.0 if i==0 else trust_records[-1]['trust'],
                                          attack_prob, anomaly, 0.95, dt=1.0)
        trust_records.append({'sample':i,'true_class':classes[int(yt)],'predicted_class':classes[int(yp)],
                              'attack_probability':attack_prob,'risk':risk,'trust':trust,'state':state})
trust_df = pd.DataFrame(trust_records)
display(trust_df.head())
if len(trust_df):
    trust_df.to_csv(RESULT_DIR/'ewata_trust_history.csv',index=False)
    plt.figure(figsize=(9,4)); plt.plot(trust_df['trust'].values)
    plt.axhline(.70,ls='--',label='Trusted threshold'); plt.axhline(.35,ls=':',label='Untrusted threshold')
    plt.ylim(0,1); plt.xlabel('Held-out device-window observation'); plt.ylabel('Trust score')
    plt.legend(); plt.grid(alpha=.25); plt.tight_layout()
    plt.savefig(RESULT_DIR/'ewata_trust_curve.png',dpi=300,bbox_inches='tight'); plt.show()

## 7. RAZTAE: Risk-Adaptive Zero-Trust Access Enforcement

The policy decision point (PDP) evaluates device identity/role, resource, operation, criticality, trust and attack severity. The policy enforcement point (PEP) returns allow, restrict, quarantine or deny. Hysteresis prevents rapid switching around decision thresholds. This is a reproducible access-policy simulation; it does not configure a real industrial gateway.

In [ ]:
def raztae_pdp(trust, attack_severity, device_role, resource, operation, criticality,
                previous_decision=None):
    trust = float(np.clip(trust,0,1))
    severity = float(np.clip(attack_severity,0,1))
    criticality = float(np.clip(criticality,0,1))
    # Attribute-based eligibility rules; adapt role/resource mappings to the deployment.
    role_allowed = {
        'operator': {'read','acknowledge'},
        'engineer': {'read','write','configure'},
        'sensor': {'publish'},
        'gateway': {'read','write','forward'}
    }
    attribute_ok = operation in role_allowed.get(str(device_role).lower(), set())
    risk = 0.55*(1-trust) + 0.30*severity + 0.15*criticality
    if not attribute_ok:
        decision = 'deny'
    elif severity >= .90 or trust < .20:
        decision = 'quarantine'
    elif trust < .40 or risk >= .65:
        decision = 'deny'
    elif trust < .70 or risk >= .40:
        decision = 'restrict'
    else:
        decision = 'allow'
    # Hysteresis: retain the prior less-permissive state near thresholds.
    rank = {'allow':0,'restrict':1,'deny':2,'quarantine':3}
    if previous_decision in rank and abs(risk-.40)<.05 and rank[previous_decision] > rank[decision]:
        decision = previous_decision
    return {'decision':decision,'risk':risk,'attribute_ok':attribute_ok,
            'reason':f'ABAC={attribute_ok}; trust={trust:.3f}; severity={severity:.3f}; criticality={criticality:.3f}'}

# Demonstration policy requests; replace with actual IIoT role/resource policy definitions.
policy_examples = [
    (0.96,0.02,'sensor','telemetry','publish',0.40),
    (0.61,0.25,'operator','dashboard','read',0.30),
    (0.28,0.82,'engineer','PLC','configure',0.95),
    (0.12,0.97,'gateway','control_network','forward',1.00),
    (0.92,0.01,'sensor','PLC','configure',0.90)
]
audit = [raztae_pdp(*x) for x in policy_examples]
audit_df = pd.DataFrame([{**dict(zip(['trust','attack_severity','device_role','resource','operation','criticality'],x)),**a} for x,a in zip(policy_examples,audit)])
display(audit_df)
audit_df.to_csv(RESULT_DIR/'raztae_access_audit.csv',index=False)

## 8. Comparative baselines and ablation study

The following baselines use the same chronological snapshot split and graph-derived device features. Graph-only removes the temporal branch; temporal-only removes graph message passing; EGRAT uses both. A lightweight MLP baseline is included as a non-graph reference. These are trained on the same training snapshots.

In [ ]:
class MLPBaseline(nn.Module):
    def __init__(self, d, c):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d,64),nn.ReLU(),nn.Dropout(.2),nn.Linear(64,c))
    def forward(self,x): return self.net(x)

class AblationNet(nn.Module):
    def __init__(self, d, c, mode='graph_only', hidden=64):
        super().__init__(); self.mode=mode
        self.sage=EdgeConditionedSAGE(d,d,hidden)
        self.temporal_in=nn.Conv1d(d,hidden,1)
        self.tcn=nn.Sequential(DS_TCN_Block(hidden,1),DS_TCN_Block(hidden,2),DS_TCN_Block(hidden,4))
        self.head=nn.Sequential(nn.Linear(hidden*(2 if mode=='full' else 1),hidden),nn.ReLU(),nn.Linear(hidden,c))
    def forward(self,x,ei,ea,h):
        if self.mode=='graph_only': z=self.sage(x,ei,ea)
        elif self.mode=='temporal_only': z=self.tcn(self.temporal_in(h))[:,:,-1]
        else: z=torch.cat([self.sage(x,ei,ea),self.tcn(self.temporal_in(h))[:,:,-1]],1)
        return self.head(z)

def fit_variant(net, epochs=8):
    net=net.to(DEVICE); opt=torch.optim.AdamW(net.parameters(),lr=2e-3,weight_decay=1e-4)
    net.train()
    for ep in range(epochs):
        for t in train_snap_ids:
            x,ei,ea,y=to_tensor_snapshot(snapshots[t]); h=history_tensor(snapshots,t)
            valid=y!=-100
            if not valid.any(): continue
            loss=criterion(net(x,ei,ea,h)[valid],y[valid])
            opt.zero_grad(); loss.backward(); opt.step()
    return net

def evaluate_variant(net):
    net.eval(); yt=[]; yp=[]
    with torch.no_grad():
        for t in test_snap_ids:
            x,ei,ea,y=to_tensor_snapshot(snapshots[t]); h=history_tensor(snapshots,t)
            valid=y!=-100
            if valid.any():
                pr=net(x,ei,ea,h)[valid].argmax(1).cpu().numpy()
                yt.extend(y[valid].cpu().numpy()); yp.extend(pr)
    if not yt: return np.nan,np.nan,np.nan
    return accuracy_score(yt,yp), precision_recall_fscore_support(yt,yp,average='weighted',zero_division=0)[0], precision_recall_fscore_support(yt,yp,average='weighted',zero_division=0)[2]

comparison = []
for name, mode in [('Graph-only','graph_only'),('Temporal-only','temporal_only'),('EGRAT full','full')]:
    net=fit_variant(AblationNet(X_all.shape[1],len(classes),mode),epochs=8)
    a,pv,f=evaluate_variant(net); comparison.append({'Model':name,'Accuracy':a,'Precision':pv,'F1':f})
comparison_df=pd.DataFrame(comparison)
display(comparison_df)
comparison_df.to_csv(RESULT_DIR/'ablation_comparison.csv',index=False)

## 9. External validation on X-IIoTID

X-IIoTID is an external dataset with a different schema. This section loads its CSV files, automatically detects common label fields, maps normal/benign labels to the normal class and other labels to attack, then aligns common feature names. The model is not retrained on X-IIoTID. A strict feature intersection is used; if the external schema does not provide the primary feature set, the notebook reports the mismatch rather than silently inventing values.

For a valid cross-dataset experiment, document the exact X-IIoTID file/version, label mapping and matched feature list in the paper.

In [ ]:
def load_external_xiiotid(folder):
    try: return read_csv_tree(folder)
    except FileNotFoundError:
        print('X-IIoTID files not found. Place extracted CSV files under', folder)
        return None

xiiot_raw = load_external_xiiotid(XIIOT_DIR)
external_result = None
if xiiot_raw is not None:
    x_label = find_col(xiiot_raw,['label','attack','attack_type','category','class'])
    if x_label is None:
        print('Could not detect X-IIoTID label column. Set x_label manually after inspecting columns.')
    else:
        xiiot = xiiot_raw.copy()
        xiiot['_label_text'] = xiiot[x_label].astype(str).str.lower()
        xiiot['_binary'] = xiiot['_label_text'].map(lambda z: 0 if z in {'normal','benign','0','false','legitimate'} else 1)
        # Only common raw feature columns are used; categorical values are factorized per external file.
        common = [c for c in feature_cols if c in xiiot.columns]
        print('Matched feature count:',len(common),'of',len(feature_cols))
        print('Matched columns:',common)
        if len(common) == len(feature_cols):
            xe = xiiot[feature_cols].copy()
            for c in xe.columns:
                xe[c] = pd.to_numeric(xe[c],errors='coerce')
            xe = xe.replace([np.inf,-np.inf],np.nan).fillna(0)
            xe = scaler.transform(xe.to_numpy(dtype=np.float32))
            # Binary external validation is supported when primary classes include normal and at least one attack class.
            # Map external binary attack to the most frequent non-normal class index for the multiclass model.
            attack_ids=[i for i,c in enumerate(classes) if c!='normal']
            if attack_ids:
                ext_y = np.where(xiiot['_binary'].to_numpy()==0,normal_id,attack_ids[0])
                ext_pred=[]
                model.eval()
                with torch.no_grad():
                    # Treat each external row as a one-node graph for schema-compatible inference.
                    for start in range(0,len(xe),2048):
                        xx=torch.tensor(xe[start:start+2048],dtype=torch.float32,device=DEVICE)
                        nxx=xx.shape[0]
                        empty_ei=torch.empty((2,0),dtype=torch.long,device=DEVICE)
                        empty_ea=torch.empty((0,xx.shape[1]),dtype=torch.float32,device=DEVICE)
                        hh=xx.unsqueeze(-1).repeat(1,1,8)
                        ext_pred.extend(model(xx,empty_ei,empty_ea,hh).argmax(1).cpu().numpy())
                ext_pred=np.asarray(ext_pred)
                ext_true_binary=(ext_y!=normal_id).astype(int)
                ext_pred_binary=(ext_pred!=normal_id).astype(int)
                external_result={'Accuracy':accuracy_score(ext_true_binary,ext_pred_binary),
                                 'Precision':precision_recall_fscore_support(ext_true_binary,ext_pred_binary,average='binary',zero_division=0)[0],
                                 'Recall':precision_recall_fscore_support(ext_true_binary,ext_pred_binary,average='binary',zero_division=0)[1],
                                 'F1':precision_recall_fscore_support(ext_true_binary,ext_pred_binary,average='binary',zero_division=0)[2]}
                display(pd.DataFrame(external_result,index=['X-IIoTID external binary']))
            else:
                print('Primary label mapping has no non-normal attack class.')
        else:
            print('External feature schema differs. Perform an explicit documented feature mapping before scoring.')

## 10. Computational efficiency, artifacts and reproducibility

This section records parameter count, model size, inference latency and the training configuration. Inference latency is measured on the current runtime and will vary with hardware, graph size and batch/window configuration.

In [ ]:
# Parameter count and serialized model size
param_count=sum(p.numel() for p in model.parameters() if p.requires_grad)
model_path=RESULT_DIR/'egrat_model.pt'
torch.save({'model_state_dict':model.state_dict(),'feature_cols':feature_cols,'classes':classes,
            'class_to_id':class_to_id,'scaler_center':getattr(scaler,'center_',None),
            'scaler_scale':getattr(scaler,'scale_',None)},model_path)
model_mb=model_path.stat().st_size/(1024**2)

# Measure one snapshot forward pass after warm-up.
latencies=[]
if snapshots:
    x,ei,ea,y=to_tensor_snapshot(snapshots[min(max(tr_end,0),len(snapshots)-1)])
    h=history_tensor(snapshots,min(max(tr_end,0),len(snapshots)-1))
    model.eval()
    with torch.no_grad():
        for _ in range(5): model(x,ei,ea,h)
        if DEVICE.type=='cuda': torch.cuda.synchronize()
        for _ in range(30):
            t0=time.perf_counter(); model(x,ei,ea,h)
            if DEVICE.type=='cuda': torch.cuda.synchronize()
            latencies.append((time.perf_counter()-t0)*1000)
efficiency={'Trainable parameters':param_count,'Saved model size (MB)':model_mb,
            'Mean snapshot inference (ms)':float(np.mean(latencies)) if latencies else np.nan,
            'Median snapshot inference (ms)':float(np.median(latencies)) if latencies else np.nan,
            'Device':str(DEVICE),'Feature count':len(feature_cols),'Device nodes':len(device_values),
            'Graph snapshots':len(snapshots)}
display(pd.DataFrame(efficiency,index=['EGRAT']).T.rename(columns={'EGRAT':'Value'}))
with open(RESULT_DIR/'run_configuration.json','w') as f:
    json.dump({'seed':SEED,'window_seconds':WINDOW_SECONDS,'max_devices':MAX_DEVICES,
               'max_windows':MAX_WINDOWS,'epochs':EPOCHS,'feature_cols':feature_cols,
               'classes':classes,'class_to_id':class_to_id,'efficiency':efficiency},f,indent=2,default=str)

print('Artifacts saved in:',RESULT_DIR.resolve())
print(sorted(p.name for p in RESULT_DIR.iterdir()))

## 11. Optional TON_IoT telemetry and system-log inspection

The TON_IoT collection contains separate network, telemetry, Linux and Windows sources. This optional cell inventories additional CSVs and attempts a device/time join when compatible timestamp and device fields are available. Keep source modality fields distinct in the final experimental protocol; do not merge unrelated records solely by row order.

In [ ]:
def inspect_auxiliary_sources(folder):
    reports=[]
    for f in Path(folder).rglob('*.csv'):
        if f.name in set(ton_raw.get('_source_file',pd.Series(dtype=str)).dropna().unique()): continue
        try:
            z=pd.read_csv(f,nrows=5,low_memory=False)
            reports.append({'file':str(f),'columns':len(z.columns),'sample_columns':', '.join(map(str,z.columns[:12]))})
        except Exception: pass
    return pd.DataFrame(reports)

aux_inventory=inspect_auxiliary_sources(TON_DIR)
display(aux_inventory if len(aux_inventory) else pd.DataFrame({'status':['No additional auxiliary CSV files detected']}))

## 12. Final run checklist

- Confirm the TON_IoT **processed network-flow** CSVs are in the primary data folder.
- Confirm the detected label, timestamp, source and destination columns match the downloaded release.
- Review class distribution and chronological split boundaries.
- Review the feature list to ensure identifiers and labels are not model inputs.
- Run the training and evaluation cells without changing the test partition.
- Place X-IIoTID CSVs in the external folder and manually map columns if schemas differ.
- Save the exact dataset versions, preprocessing settings, random seed, and hardware details with the reported results.

**Research interpretation:** EGRAT is the hybrid spatial-temporal detector; EWATA converts detector and behavioural evidence into time-adaptive device trust; RAZTAE maps trust and contextual risk to ABAC/RBAC-informed access decisions. The notebook simulates the policy enforcement decision and audit record; integration with a production gateway requires deployment-specific adapters and safety approval.